Encuentra la carpeta de datos desde el notebook

In [4]:
from pathlib import Path
import pandas as pd

actual = Path.cwd().resolve()
candidatas = [actual, *actual.parents]
ROOT = next(
    (ruta for ruta in candidatas if (ruta / "data" / "bronze").is_dir()),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "No encontré data/bronze. Abre el notebook desde el proyecto "
        "o ejecuta primero el pipeline."
    )

BRONZE = ROOT / "data" / "bronze"
archivos = sorted(BRONZE.glob("*.parquet"))

if not archivos:
    raise FileNotFoundError(
        f"No hay archivos Parquet en {BRONZE}. Ejecuta python -m src.pipeline."
    )

print(f"Proyecto: {ROOT}")
print(f"Parquet encontrados: {len(archivos)}")
for archivo in archivos:
    print("-", archivo.name)

Proyecto: /workspaces/observatorio-energia
Parquet encontrados: 10
- enerandes_ventas_2023.parquet
- enerandes_ventas_2024.parquet
- enerandes_ventas_2025.parquet
- enerandes_ventas_2026.parquet
- ideam_precipitacion_diaria.parquet
- noaa_oni.parquet
- trm_colombia.parquet
- xm_demanda.parquet
- xm_embalses.parquet
- xm_precio_bolsa.parquet


Ejecuta específicamente las líneas de fechas
En otra celda, carga IDEAM en df y luego calcula las fechas:

In [5]:
df = pd.read_parquet(BRONZE / "ideam_precipitacion_diaria.parquet")

fechas = pd.to_datetime(df["fecha"], errors="coerce")
print("Fecha inicial:", fechas.min())
print("Fecha final:", fechas.max())
print("Fechas que no se pudieron interpretar:", fechas.isna().sum())

Fecha inicial: 2026-06-01 00:00:00
Fecha final: 2026-07-24 00:00:00
Fechas que no se pudieron interpretar: 0


La conversión con errors="coerce" permite contar fechas inválidas en vez de detener la celda con un error. Si quieres probar XM, la columna se llama Date —con mayúscula—, no fecha.

Perfilamiento completo de los Parquet
Esta celda presenta filas, columnas, tipos, nulos, duplicados y rangos cuando encuentra columnas con fechas reconocibles:

In [6]:
from IPython.display import display

columnas_fecha = {"date", "fecha", "vigenciadesde", "vigenciahasta"}

for archivo in archivos:
    df = pd.read_parquet(archivo)

    print(f"\n## {archivo.name}")
    print(f"Filas: {len(df):,} | Columnas: {len(df.columns)}")
    print(f"Filas duplicadas exactas: {df.duplicated().sum():,}")

    perfil_columnas = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    })
    display(perfil_columnas)

    fechas_archivo = [
        columna for columna in df.columns
        if str(columna).lower() in columnas_fecha
    ]

    if fechas_archivo:
        for columna in fechas_archivo:
            fechas = pd.to_datetime(df[columna], errors="coerce")
            print(
                f"{columna}: {fechas.min()} a {fechas.max()} "
                f"| fechas inválidas: {fechas.isna().sum()}"
            )
    else:
        print("No se encontró una columna de fecha con nombre reconocido.")


## enerandes_ventas_2023.parquet
Filas: 1,114 | Columnas: 10
Filas duplicadas exactas: 32


,tipo,nulos,porcentaje_nulos
col_0,str,0,0.00
col_1,str,1,0.09
col_2,str,16,1.44
col_3,str,1,0.09
col_4,str,10,0.90
col_5,str,8,0.72
col_6,str,1,0.09
_fuente,str,0,0.00
_origen,str,0,0.00
_extraido_en,str,0,0.00


No se encontró una columna de fecha con nombre reconocido.

## enerandes_ventas_2024.parquet
Filas: 1,114 | Columnas: 11
Filas duplicadas exactas: 32


,tipo,nulos,porcentaje_nulos
col_0,str,0,0.00
col_1,str,1,0.09
col_2,str,4,0.36
col_3,str,1,0.09
col_4,str,8,0.72
col_5,str,1,0.09
col_6,str,12,1.08
col_7,str,1,0.09
_fuente,str,0,0.00
_origen,str,0,0.00


No se encontró una columna de fecha con nombre reconocido.

## enerandes_ventas_2025.parquet
Filas: 1,117 | Columnas: 10
Filas duplicadas exactas: 32


...

Date: 2024-01-01 00:00:00 a 2026-09-30 00:00:00 | fechas inválidas: 0

## xm_embalses.parquet
Filas: 1,004 | Columnas: 6
Filas duplicadas exactas: 0


,tipo,nulos,porcentaje_nulos
Date,str,0,0.0
Id,str,0,0.0
Value,str,0,0.0
_fuente,str,0,0.0
_origen,str,0,0.0
_extraido_en,str,0,0.0


Date: 2024-01-01 00:00:00 a 2026-09-30 00:00:00 | fechas inválidas: 0

## xm_precio_bolsa.parquet
Filas: 1,004 | Columnas: 30
Filas duplicadas exactas: 0


,tipo,nulos,porcentaje_nulos
Date,str,0,0.0
Id,str,0,0.0
code,str,0,0.0
Hour01,str,0,0.0
Hour02,str,0,0.0
Hour03,str,0,0.0
Hour04,str,0,0.0
Hour05,str,0,0.0
Hour06,str,0,0.0
Hour07,str,0,0.0


Date: 2024-01-01 00:00:00 a 2026-09-30 00:00:00 | fechas inválidas: 0


En EnerAndes las columnas vienen como col_0, col_1, etc.; por eso este código no asume que col_0 sea siempre una fecha. En esos archivos también hay encabezados y títulos dentro de las filas, así que hay que interpretarlos antes de calcular un rango confiable.

Reproduce los hallazgos numéricos
Esta celda vuelve a calcular los hallazgos reportados, usando los archivos que encuentre localmente:

In [7]:
# EnerAndes: duplicados y nulos en columnas de datos col_*
enerandes = sorted(BRONZE.glob("enerandes_ventas_*.parquet"))

if enerandes:
    filas_total = 0
    duplicados_total = 0
    nulos_total = 0

    for archivo in enerandes:
        df = pd.read_parquet(archivo)
        columnas_datos = [c for c in df.columns if c.startswith("col_")]

        filas_total += len(df)
        duplicados_total += int(df.duplicated().sum())
        nulos_total += int(df[columnas_datos].isna().sum().sum())

    print(
        f"EnerAndes: {duplicados_total:,} duplicados de {filas_total:,} filas "
        f"({duplicados_total / filas_total * 100:.2f} %)"
    )
    print(f"EnerAndes: {nulos_total:,} celdas nulas en columnas de datos")

    for archivo in enerandes:
        df = pd.read_parquet(archivo)
        columnas_datos = [c for c in df.columns if c.startswith("col_")]
        print(
            f"{archivo.name}: {len(df):,} filas, "
            f"{len(columnas_datos)} columnas de datos; "
            f"primeros valores de col_0: {df['col_0'].head(3).tolist()}"
        )

# IDEAM: cobertura de fechas y conteos de observaciones
archivo_ideam = BRONZE / "ideam_precipitacion_diaria.parquet"

if archivo_ideam.exists():
    ideam = pd.read_parquet(archivo_ideam)
    fechas = pd.to_datetime(ideam["fecha"], errors="coerce").dropna().dt.normalize()

    inicio = pd.Timestamp("2026-06-01")
    fin = pd.Timestamp("2026-09-30")
    dias_solicitados = pd.date_range(inicio, fin, freq="D")
    dias_observados = pd.DatetimeIndex(fechas.unique())
    dias_faltantes = dias_solicitados.difference(dias_observados)

    print(
        f"IDEAM: {len(dias_observados)} fechas observadas; "
        f"{len(dias_faltantes)} días sin registros de "
        f"{len(dias_solicitados)} solicitados"
    )

    n_obs = pd.to_numeric(ideam["n_obs"], errors="coerce")
    sobre_144 = int((n_obs > 144).sum())

    print(
        f"IDEAM: {sobre_144:,} de {len(ideam):,} registros tienen n_obs > 144; "
        f"máximo={n_obs.max():,.0f}; mediana={n_obs.median():,.0f}"
    )

EnerAndes: 120 duplicados de 4,184 filas (2.87 %)
EnerAndes: 148 celdas nulas en columnas de datos
enerandes_ventas_2023.parquet: 1,114 filas, 7 columnas de datos; primeros valores de col_0: ['Fecha', '2023-04', '2023-09']
enerandes_ventas_2024.parquet: 1,114 filas, 8 columnas de datos; primeros valores de col_0: ['PERIODO', 'jun-24', '09/2024']
enerandes_ventas_2025.parquet: 1,117 filas, 7 columnas de datos; primeros valores de col_0: ['EnerAndes S.A.S. E.S.P. - Reporte de ventas 2025 - Gerencia Comercial', 'Fuente: sistema comercial. Uso interno.', nan]
enerandes_ventas_2026.parquet: 839 filas, 7 columnas de datos; primeros valores de col_0: ['EnerAndes S.A.S. E.S.P. - Reporte de ventas 2026 - Gerencia Comercial', 'Fuente: sistema comercial. Uso interno.', nan]
IDEAM: 52 fechas observadas; 70 días sin registros de 122 solicitados
IDEAM: 149 de 2,784 registros tienen n_obs > 144; máximo=16,302; mediana=144
